# Building Faster Transformers with Triton

## Part 1 — Your First Triton Kernel

In the slides, we introduced this mental model:

> **program ID → offsets → load → compute → store**

We will now use that pattern to implement vector addition on the GPU.

### By the end of this section, you should be able to

- explain how Triton divides a vector across multiple programs;
- calculate the elements handled by one program;
- use a mask to protect out-of-bounds memory accesses;
- check a custom kernel against PyTorch; and
- benchmark both implementations on the same GPU.

> **Important:** Select a GPU runtime in Colab before continuing:  
> **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU).

---
## 0. Setup

The next cell installs Triton and imports the libraries used in this tutorial. The installation and environment checks are setup code and are not part of the kernel programming exercise.

In [ ]:
%pip install -q triton

In [ ]:
import pandas as pd
import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), (
    "No GPU was detected. In Colab, select Runtime → Change runtime type → GPU, "
    "then run the notebook again."
)

DEVICE = triton.runtime.driver.active.get_active_torch_device()
torch.manual_seed(0)

print(f"GPU:    {torch.cuda.get_device_name(DEVICE)}")
print(f"PyTorch: {torch.__version__}")
print(f"Triton:  {triton.__version__}")

---
## 1. Start with a trusted reference

Before writing a custom kernel, define what the operation is supposed to compute.

For two vectors $x$ and $y$ of length $N$:

$$
\text{output}[i] = x[i] + y[i], \qquad 0 \le i < N
$$

PyTorch already provides a correct GPU implementation. We will use it as our reference.

In [ ]:
N = 10
x = torch.arange(N, device=DEVICE, dtype=torch.float32)
y = torch.full((N,), 10.0, device=DEVICE)

expected = x + y


print("x:       ", x)
print("y:       ", y)
print("x + y:   ", expected)

---
## 2. How should we divide the work?

Suppose each Triton program handles `BLOCK_SIZE = 4` elements.

```text
Vector length = 10

Program 0        Program 1        Program 2
[0, 1, 2, 3]    [4, 5, 6, 7]    [8, 9, ?, ?]
```

The last program contains two valid positions and two positions beyond the end of the vector. We therefore need two things:

1. **Offsets** — the element indices handled by the current program.
2. **A mask** — which offsets are valid.

The formulas are:

```python
pid = program ID

offsets = pid * BLOCK_SIZE + [0, 1, ..., BLOCK_SIZE - 1]
mask = offsets < N
```

Run the following Python simulation before looking at the Triton kernel.

In [ ]:
N = 10
BLOCK_SIZE = 4
number_of_programs = triton.cdiv(N, BLOCK_SIZE)

for pid in range(number_of_programs):
    offsets = pid * BLOCK_SIZE + torch.arange(BLOCK_SIZE)
    mask = offsets < N
    print(f"program {pid}: offsets={offsets.tolist()}  mask={mask.tolist()}")

### Check your understanding

For `N = 17` and `BLOCK_SIZE = 8`:

- How many programs are launched?
- Which offsets belong to program 2?
- What is the mask for program 2?

Predict the answers before running the next cell.

In [ ]:
N = 17
BLOCK_SIZE = 8
number_of_programs = triton.cdiv(N, BLOCK_SIZE)

pid = 2
offsets = pid * BLOCK_SIZE + torch.arange(BLOCK_SIZE)
mask = offsets < N

print("number of programs:", number_of_programs)
print("program 2 offsets: ", offsets.tolist())
print("program 2 mask:    ", mask.tolist())

---
## 3. The Triton programming pattern

A Triton kernel is executed by many **program instances**. Each program uses its ID to determine which part of the tensor it owns.

The vector-add kernel follows six steps:

```text
1. Get the program ID
2. Calculate this program's offsets
3. Create a boundary mask
4. Load x and y
5. Add them
6. Store the result
```

Read the complete kernel once. Do not try to memorize it.

In [ ]:
@triton.jit
def vector_add_kernel(
    x_ptr,                         # Pointer to the first input vector
    y_ptr,                         # Pointer to the second input vector
    output_ptr,                    # Pointer to the output vector
    n_elements,                    # Number of valid elements
    BLOCK_SIZE: tl.constexpr,      # Elements handled by one program
):
    # 1. Which program instance is running?
    pid = tl.program_id(axis=0)

    # 2. Which vector positions belong to this program?
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)

    # 3. Which positions are inside the vector?
    mask = offsets < n_elements

    # 4. Load input values from global memory.
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)

    # 5. Compute a block of additions.
    output = x + y

    # 6. Store valid results back to global memory.
    tl.store(output_ptr + offsets, output, mask=mask)

### What each important line means

| Triton expression | Meaning |
|---|---|
| `tl.program_id(axis=0)` | Identify the current program along the first launch dimension |
| `tl.arange(0, BLOCK_SIZE)` | Create a block of local indices |
| `x_ptr + offsets` | Form pointers to the elements this program will load |
| `tl.load(..., mask=mask)` | Load only valid positions |
| `tl.store(..., mask=mask)` | Store only valid positions |

Notice that `x`, `y`, `offsets`, and `mask` each represent a **block of values**, not one scalar value.

---
## 4. Launching the kernel

The kernel describes what **one program** does. The Python wrapper decides:

- how many output elements exist;
- how many programs to launch; and
- where the output will be stored.

For a vector of length `N`, the number of programs is:

```python
ceil(N / BLOCK_SIZE)
```

`triton.cdiv` performs this ceiling division.

In [ ]:
def vector_add(x: torch.Tensor, y: torch.Tensor, block_size: int = 256):
    assert x.shape == y.shape
    assert x.device == y.device == DEVICE

    output = torch.empty_like(x)
    n_elements = output.numel()

    # One-dimensional launch grid: one program per block of elements.
    grid = (triton.cdiv(n_elements, block_size),)

    vector_add_kernel[grid](
        x,
        y,
        output,
        n_elements,
        BLOCK_SIZE=block_size,
    )

    return output

---
## 5. Correctness first

Always check correctness before measuring speed.

We deliberately use `100_003` elements, which is not divisible by `256`. This tests whether the mask handles the final partial block correctly.

In [ ]:
N = 100_003
x = torch.randn(N, device=DEVICE)
y = torch.randn(N, device=DEVICE)

actual = vector_add(x, y, block_size=256)
expected = x + y

torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)

max_error = (actual - expected).abs().max().item()
print(f"✓ Complete output matches PyTorch")
print(f"Maximum absolute error: {max_error:.3e}")

---
## 6. Participant exercise: complete the indexing

Complete only the two missing lines:

1. Calculate `offsets` from `pid`, `BLOCK_SIZE`, and `tl.arange`.
2. Create a mask that is `True` only when an offset is smaller than `n_elements`.

Do not change the load, addition, or store operations.

In [ ]:
@triton.jit
def vector_add_exercise_kernel(
    x_ptr,
    y_ptr,
    output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(axis=0)

    # TODO 1: Which offsets belong to this program?
    offsets = ...

    # TODO 2: Which offsets are valid?
    mask = ...

    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    output = x + y
    tl.store(output_ptr + offsets, output, mask=mask)

In [ ]:
def vector_add_exercise(x: torch.Tensor, y: torch.Tensor, block_size: int = 256):
    output = torch.empty_like(x)
    n_elements = output.numel()
    grid = (triton.cdiv(n_elements, block_size),)

    vector_add_exercise_kernel[grid](
        x,
        y,
        output,
        n_elements,
        BLOCK_SIZE=block_size,
    )
    return output

# Run this after completing the two TODOs above.
N = 10_003
x_test = torch.randn(N, device=DEVICE)
y_test = torch.randn(N, device=DEVICE)

exercise_output = vector_add_exercise(x_test, y_test)
torch.testing.assert_close(exercise_output, x_test + y_test, rtol=1e-5, atol=1e-6)
print("✓ Exercise kernel is correct")

<details>
<summary><strong>Reveal the two missing lines</strong></summary>

```python
offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
mask = offsets < n_elements
```

</details>

---
## 7. Benchmark against PyTorch

Both implementations run on the **same GPU** and compute the same operation.

Vector addition performs little arithmetic relative to the amount of data it reads and writes. It is therefore primarily a **memory-bandwidth-bound** operation.

For each output element, the kernel moves approximately:

- one value from `x`;
- one value from `y`; and
- one value into `output`.

The benchmark reports effective memory bandwidth in GB/s.

In [ ]:
def benchmark_ms(fn):
    """Return median execution time in milliseconds."""
    return triton.testing.do_bench(fn)


def effective_bandwidth_gbps(x: torch.Tensor, time_ms: float):
    # Two input reads + one output write.
    bytes_moved = 3 * x.numel() * x.element_size()
    return bytes_moved / (time_ms * 1e-3) / 1e9


N = 2**24
x_bench = torch.randn(N, device=DEVICE)
y_bench = torch.randn(N, device=DEVICE)

# Warm up both implementations before timing.
_ = vector_add(x_bench, y_bench)
_ = x_bench + y_bench
torch.cuda.synchronize()

triton_ms = benchmark_ms(lambda: vector_add(x_bench, y_bench))
torch_ms = benchmark_ms(lambda: x_bench + y_bench)

results = pd.DataFrame([
    {
        "Implementation": "Triton",
        "Time (ms)": triton_ms,
        "Effective bandwidth (GB/s)": effective_bandwidth_gbps(x_bench, triton_ms),
    },
    {
        "Implementation": "PyTorch",
        "Time (ms)": torch_ms,
        "Effective bandwidth (GB/s)": effective_bandwidth_gbps(x_bench, torch_ms),
    },
])

results.style.format({
    "Time (ms)": "{:.3f}",
    "Effective bandwidth (GB/s)": "{:.1f}",
})

### Interpreting the result

Do not expect the beginner Triton kernel to defeat PyTorch by a large margin. PyTorch's elementwise addition is already highly optimized.

The important result is that we expressed the GPU computation ourselves using the same pattern we will reuse later:

> **program ID → offsets → mask → load → compute → store**

Performance numbers also vary with GPU model, tensor size, dtype, and background activity.

---
## 8. Small experiment: does block size matter?

Change how many elements each Triton program handles, then measure the result.

Before running the cell, predict whether the largest block size will always be the fastest.

In [ ]:
rows = []

for block_size in [64, 128, 256, 512, 1024]:
    # Verify correctness for every configuration before recording its speed.
    output = vector_add(x_bench, y_bench, block_size=block_size)
    torch.testing.assert_close(output, x_bench + y_bench, rtol=1e-5, atol=1e-6)

    time_ms = benchmark_ms(
        lambda bs=block_size: vector_add(x_bench, y_bench, block_size=bs)
    )

    rows.append({
        "BLOCK_SIZE": block_size,
        "Time (ms)": time_ms,
        "Effective bandwidth (GB/s)": effective_bandwidth_gbps(x_bench, time_ms),
    })

block_size_results = pd.DataFrame(rows)
block_size_results.style.format({
    "Time (ms)": "{:.3f}",
    "Effective bandwidth (GB/s)": "{:.1f}",
})

### Discuss

- Did all block sizes produce the correct output?
- Which block size was fastest on your GPU?
- Was the largest block size always best?
- Why should performance decisions be measured rather than guessed?

The purpose of this experiment is not to find a universal best block size. It is to observe that launch choices influence how efficiently the GPU executes a kernel.

---
## 9. What to remember

1. **A Triton kernel runs as many program instances.**
2. **The program ID identifies which block of data one program owns.**
3. **Offsets turn that program ID into tensor positions.**
4. **Masks protect the final partial block.**
5. **Correctness comes before benchmarking.**
6. **Performance must be measured on the target GPU.**

### Next: from independent elements to data reuse

Vector addition loads each value once, uses it once, and stores one result. There is little opportunity for reuse.

Matrix multiplication is different: the same values from matrices $A$ and $B$ contribute to many output values. In the next section, we will use **tiling** to load blocks of data and reuse them across multiple calculations.

---
### Reference

This exercise is adapted for an introductory classroom setting from the official Triton vector-addition tutorial:  
https://triton-lang.org/main/getting-started/tutorials/01-vector-add.html

---
# Part 2 — From Vector Addition to Tiled Matrix Multiplication

## Learning outcome

By the end of this section, you should be able to:

- explain why matrix multiplication offers opportunities for data reuse;
- map a two-dimensional Triton program grid to tiles of an output matrix;
- read the main stages of a tiled matrix-multiplication kernel;
- modify the tile shape; and
- check correctness and benchmark against GPU baselines.

We are **not** trying to reproduce every optimization inside a production GEMM library.  
We are learning the design pattern that makes optimized matrix multiplication possible.

---
## 10. Why matrix multiplication is different

For matrices

\[
A \in \mathbb{R}^{M \times K},
\qquad
B \in \mathbb{R}^{K \times N},
\]

matrix multiplication produces

\[
C = AB,
\qquad
C_{ij} = \sum_{k=0}^{K-1} A_{ik} B_{kj}.
\]


One output value uses:

- one row of \(A\); and
- one column of \(B\).

But neighbouring output values reuse much of the same data:

- values in one row of \(A\) contribute to many columns of \(C\);
- values in one column of \(B\) contribute to many rows of \(C\).

A naive implementation may repeatedly fetch those values from global memory.

> **Tiling changes the unit of work from one output value to one output tile.**

In [ ]:
# A small trusted reference.

A_small = torch.tensor(
    [[1, 2, 3],
     [4, 5, 6]],
    device=DEVICE,
    dtype=torch.float16,
)

B_small = torch.tensor(
    [[1, 2],
     [3, 4],
     [5, 6]],
    device=DEVICE,
    dtype=torch.float16,
)

C_small = A_small @ B_small

print("A shape:", tuple(A_small.shape))
print("B shape:", tuple(B_small.shape))
print("C = A @ B:")
print(C_small)

---
## 11. A deliberately naive GPU formulation

The following PyTorch expression computes matrix multiplication by:

1. multiplying every \(A_{ik}\) with every \(B_{kj}\);
2. materializing an intermediate tensor of shape \([M, K, N]\); and
3. reducing that tensor across \(K\).

It is mathematically valid, but it moves and stores far more data than necessary.

We use it only as a teaching baseline.

In [ ]:
def naive_broadcast_matmul(a: torch.Tensor, b: torch.Tensor):
    """Teaching baseline: correct, GPU-based, but intentionally memory hungry."""
    assert a.ndim == 2 and b.ndim == 2
    assert a.shape[1] == b.shape[0]

    # [M, K, 1] * [1, K, N] -> [M, K, N]
    products = a.float()[:, :, None] * b.float()[None, :, :]

    # Reduce K and return the same output dtype as the inputs.
    return products.sum(dim=1).to(a.dtype)


naive_small = naive_broadcast_matmul(A_small, B_small)
torch.testing.assert_close(naive_small, C_small, rtol=1e-3, atol=1e-3)

M, K = A_small.shape
_, N = B_small.shape

print("Intermediate shape:", (M, K, N))
print("Output shape:      ", tuple(naive_small.shape))
print("✓ Naive GPU formulation is correct")

### Why this formulation is inefficient

The output contains \(M \times N\) values.

The broadcast intermediate contains \(M \times K \times N\) values.

For realistic matrix sizes, that intermediate can be much larger than the output. An efficient matrix-multiplication kernel does not materialize it. Instead, it accumulates partial sums close to the compute units.

In [ ]:
rows = []

for size in [128, 256, 512, 1024]:
    output_elements = size * size
    intermediate_elements = size * size * size

    rows.append({
        "M = K = N": size,
        "Output elements": output_elements,
        "Naive intermediate elements": intermediate_elements,
        "FP32 intermediate (MiB)": intermediate_elements * 4 / 2**20,
    })

pd.DataFrame(rows).style.format({
    "Output elements": "{:,.0f}",
    "Naive intermediate elements": "{:,.0f}",
    "FP32 intermediate (MiB)": "{:,.1f}",
})

---
## 12. Map Triton programs to output tiles

Vector addition used a one-dimensional grid.

Matrix multiplication uses a two-dimensional grid:

```text
program (pid_m, pid_n) → one BLOCK_M × BLOCK_N tile of C
```

`pid_m` chooses a block of output rows.  
`pid_n` chooses a block of output columns.

In [ ]:
M, N = 70, 50
BLOCK_M, BLOCK_N = 32, 32

programs_m = triton.cdiv(M, BLOCK_M)
programs_n = triton.cdiv(N, BLOCK_N)

print(f"Grid: {programs_m} × {programs_n} programs\n")

for pid_m in range(programs_m):
    for pid_n in range(programs_n):
        row_start = pid_m * BLOCK_M
        row_end = min(row_start + BLOCK_M, M)

        col_start = pid_n * BLOCK_N
        col_end = min(col_start + BLOCK_N, N)

        print(
            f"program ({pid_m}, {pid_n}) -> "
            f"rows [{row_start}:{row_end}), "
            f"columns [{col_start}:{col_end})"
        )

### The tiled computation

One program:

1. chooses one tile of \(C\);
2. loads one tile of \(A\) and one tile of \(B\);
3. multiplies and accumulates them;
4. moves to the next pair of tiles along \(K\); and
5. stores the completed output tile.

```text
choose C tile
      ↓
load A tile and B tile
      ↓
multiply and accumulate
      ↓
move along K
      ↓
store C tile
```

---
## 13. Read a tiled Triton matrix-multiplication kernel

Read the kernel in five stages rather than trying to understand every pointer expression at once.

Focus on these questions:

- Which output tile does this program own?
- Which rows of \(A\) and columns of \(B\) are needed?
- Where are partial results accumulated?
- Why do the loads and final store need masks?

In [ ]:
@triton.jit
def tiled_matmul_kernel(
    a_ptr,
    b_ptr,
    c_ptr,
    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    # Stage 1: Which output tile does this program own?
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    offsets_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offsets_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offsets_k = tl.arange(0, BLOCK_K)

    # Stage 2: Keep partial sums close to the compute units.
    accumulator = tl.zeros(
        (BLOCK_M, BLOCK_N),
        dtype=tl.float32,
    )

    # Stage 3: Walk through the reduction dimension in BLOCK_K chunks.
    for k_start in range(0, K, BLOCK_K):
        a_offsets = (
            offsets_m[:, None] * stride_am
            + (k_start + offsets_k[None, :]) * stride_ak
        )

        b_offsets = (
            (k_start + offsets_k[:, None]) * stride_bk
            + offsets_n[None, :] * stride_bn
        )

        a_mask = (
            (offsets_m[:, None] < M)
            & (k_start + offsets_k[None, :] < K)
        )

        b_mask = (
            (k_start + offsets_k[:, None] < K)
            & (offsets_n[None, :] < N)
        )

        a_tile = tl.load(
            a_ptr + a_offsets,
            mask=a_mask,
            other=0.0,
        )

        b_tile = tl.load(
            b_ptr + b_offsets,
            mask=b_mask,
            other=0.0,
        )

        # Each loaded value contributes to several output values.
        accumulator += tl.dot(a_tile, b_tile)

    # Stage 4: Calculate addresses for the completed output tile.
    c_offsets = (
        offsets_m[:, None] * stride_cm
        + offsets_n[None, :] * stride_cn
    )

    c_mask = (
        (offsets_m[:, None] < M)
        & (offsets_n[None, :] < N)
    )

    # Stage 5: Store only valid output positions.
    tl.store(
        c_ptr + c_offsets,
        accumulator,
        mask=c_mask,
    )

### Kernel reading guide

| Stage | Main Triton expressions | Meaning |
|---|---|---|
| Choose output tile | `tl.program_id`, `offsets_m`, `offsets_n` | Map one program to rows and columns of \(C\) |
| Create accumulator | `tl.zeros((BLOCK_M, BLOCK_N))` | Hold partial output values |
| Load input tiles | `tl.load(..., mask=..., other=0.0)` | Safely fetch blocks of \(A\) and \(B\) |
| Multiply and accumulate | `tl.dot(a_tile, b_tile)` | Reuse loaded values across many multiply-adds |
| Store result | `tl.store(..., mask=c_mask)` | Write one completed output tile |

---
## 14. Launch the two-dimensional kernel

The wrapper supplies:

- the matrix dimensions;
- tensor strides;
- the tile shape; and
- a two-dimensional launch grid.

Tile sizes are compile-time constants. We will modify them later.

In [ ]:
def tiled_matmul(
    a: torch.Tensor,
    b: torch.Tensor,
    block_m: int = 32,
    block_n: int = 32,
    block_k: int = 32,
):
    assert a.ndim == 2 and b.ndim == 2
    assert a.shape[1] == b.shape[0]
    assert a.device == b.device == DEVICE
    assert a.dtype == b.dtype == torch.float16

    M, K = a.shape
    _, N = b.shape

    output = torch.empty(
        (M, N),
        device=a.device,
        dtype=a.dtype,
    )

    grid = (
        triton.cdiv(M, block_m),
        triton.cdiv(N, block_n),
    )

    tiled_matmul_kernel[grid](
        a,
        b,
        output,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        output.stride(0),
        output.stride(1),
        BLOCK_M=block_m,
        BLOCK_N=block_n,
        BLOCK_K=block_k,
        num_warps=4,
    )

    return output

---
## 15. Correctness first—especially at tile boundaries

We deliberately choose dimensions that are not all divisible by the tile sizes.

The masks should:

- replace invalid input positions with zero; and
- prevent invalid output stores.

In [ ]:
M, K, N = 130, 96, 150

a_test = torch.randn(
    (M, K),
    device=DEVICE,
    dtype=torch.float16,
)

b_test = torch.randn(
    (K, N),
    device=DEVICE,
    dtype=torch.float16,
)

actual = tiled_matmul(a_test, b_test)
expected = torch.matmul(a_test, b_test)

torch.testing.assert_close(
    actual,
    expected,
    rtol=2e-2,
    atol=2e-2,
)

max_error = (actual - expected).abs().max().item()

print("Output shape:", tuple(actual.shape))
print(f"Maximum absolute error: {max_error:.6f}")
print("✓ Tiled matrix multiplication is correct")

---
## 16. Compare three GPU implementations

We compare:

1. **Naive GPU composition:** broadcast, materialize \([M,K,N]\), then reduce.
2. **Tiled Triton:** load tiles and accumulate without materializing that intermediate.
3. **`torch.matmul`:** a trusted, highly optimized GPU baseline.

All three run on the same GPU and compute the same matrix product.

We use a moderate size for the naive implementation because its intermediate grows cubically.

In [ ]:
def matmul_tflops(M: int, N: int, K: int, time_ms: float):
    operations = 2 * M * N * K
    return operations / (time_ms * 1e-3) / 1e12


M = K = N = 256

a_bench_small = torch.randn(
    (M, K),
    device=DEVICE,
    dtype=torch.float16,
)

b_bench_small = torch.randn(
    (K, N),
    device=DEVICE,
    dtype=torch.float16,
)

reference_small = torch.matmul(a_bench_small, b_bench_small)

implementations = {
    "Naive GPU composition": lambda: naive_broadcast_matmul(
        a_bench_small,
        b_bench_small,
    ),
    "Tiled Triton": lambda: tiled_matmul(
        a_bench_small,
        b_bench_small,
    ),
    "torch.matmul": lambda: torch.matmul(
        a_bench_small,
        b_bench_small,
    ),
}

comparison_rows = []

for name, fn in implementations.items():
    output = fn()

    torch.testing.assert_close(
        output,
        reference_small,
        rtol=2e-2,
        atol=2e-2,
    )

    time_ms = benchmark_ms(fn)

    comparison_rows.append({
        "Implementation": name,
        "Correct": "✓",
        "Time (ms)": time_ms,
        "TFLOP/s": matmul_tflops(M, N, K, time_ms),
    })

comparison = pd.DataFrame(comparison_rows)

comparison.style.format({
    "Time (ms)": "{:.3f}",
    "TFLOP/s": "{:.2f}",
})

### Interpreting the comparison

The purpose is not to claim that this introductory Triton kernel is universally faster than `torch.matmul`.

`torch.matmul` uses mature vendor libraries with hardware-specific optimizations. Our result depends on:

- GPU model;
- input size and dtype;
- tile shape;
- compiler version; and
- library version.

The important observation is that the naive formulation performs the same mathematics while creating much more memory traffic.

---
## 17. Participant exercise: change the tile shape

Keep the kernel unchanged. Modify only:

- `BLOCK_M`;
- `BLOCK_N`; and
- `BLOCK_K`.

Try these configurations:

```text
16 × 16 × 16
32 × 32 × 16
32 × 32 × 32
64 × 32 × 32
```

Before benchmarking each configuration:

1. check the complete output against `torch.matmul`;
2. record the runtime; and
3. compare the measured result with your prediction.

In [ ]:
M = K = N = 1024

a_tiles = torch.randn(
    (M, K),
    device=DEVICE,
    dtype=torch.float16,
)

b_tiles = torch.randn(
    (K, N),
    device=DEVICE,
    dtype=torch.float16,
)

reference_tiles = torch.matmul(a_tiles, b_tiles)

tile_configs = [
    (16, 16, 16),
    (32, 32, 16),
    (32, 32, 32),
    (64, 32, 32),
]

tile_rows = []

for block_m, block_n, block_k in tile_configs:
    fn = lambda bm=block_m, bn=block_n, bk=block_k: tiled_matmul(
        a_tiles,
        b_tiles,
        block_m=bm,
        block_n=bn,
        block_k=bk,
    )

    output = fn()

    torch.testing.assert_close(
        output,
        reference_tiles,
        rtol=2e-2,
        atol=2e-2,
    )

    time_ms = benchmark_ms(fn)

    tile_rows.append({
        "BLOCK_M": block_m,
        "BLOCK_N": block_n,
        "BLOCK_K": block_k,
        "Correct": "✓",
        "Time (ms)": time_ms,
        "TFLOP/s": matmul_tflops(M, N, K, time_ms),
    })

tile_results = pd.DataFrame(tile_rows)

tile_results.style.format({
    "Time (ms)": "{:.3f}",
    "TFLOP/s": "{:.2f}",
})

### Discuss

- Did every tile shape produce the correct result?
- Which configuration was fastest on your GPU?
- Was the largest tile always best?
- What might make a tile too small?
- What might make a tile too large?
- Why must tile selection be measured rather than guessed?

> **Core lesson:** tiling creates reuse, but the best tile shape depends on the hardware and workload.

---
# Part 3 — FlashAttention as an Application of Tiling

We will not write a full FlashAttention kernel during this introductory session.

Instead, we will use it as a case study:

> How can the same “load tiles, reuse data, avoid large intermediates” idea accelerate transformer attention?

---
## 18. Where standard attention moves data

Scaled dot-product attention is:

\[
S = \frac{QK^\top}{\sqrt{d}}, \qquad
P = \operatorname{softmax}(S), \qquad
O = PV.
\]

A straightforward implementation may:

```text
load Q and K
     ↓
compute the N × N score matrix
     ↓
write scores to global memory
     ↓
read scores for softmax
     ↓
write probabilities
     ↓
read probabilities for P @ V
```

The score and probability matrices grow quadratically with sequence length.

In [ ]:
def attention_matrix_memory_mib(
    batch: int,
    heads: int,
    sequence_length: int,
    bytes_per_element: int = 2,
):
    elements = batch * heads * sequence_length * sequence_length
    return elements * bytes_per_element / 2**20


memory_rows = []

for sequence_length in [512, 1024, 2048, 4096, 8192]:
    one_matrix_mib = attention_matrix_memory_mib(
        batch=1,
        heads=8,
        sequence_length=sequence_length,
        bytes_per_element=2,
    )

    memory_rows.append({
        "Sequence length": sequence_length,
        "One FP16 score matrix (MiB)": one_matrix_mib,
        "Scores + probabilities (MiB)": 2 * one_matrix_mib,
    })

attention_memory = pd.DataFrame(memory_rows)

attention_memory.style.format({
    "One FP16 score matrix (MiB)": "{:,.1f}",
    "Scores + probabilities (MiB)": "{:,.1f}",
})

---
## 19. FlashAttention applies the tiled-kernel pattern

Conceptually:

```text
for each Q tile:
    load the Q tile

    for each K/V tile:
        load K and V tiles
        compute partial attention scores
        update softmax statistics
        accumulate partial output

    store the completed output tile
```

The full \(N \times N\) attention matrix is not materialized in global memory.

The connection to tiled matrix multiplication is direct:

| Tiled matrix multiplication | FlashAttention |
|---|---|
| Load \(A\) and \(B\) tiles | Load \(Q\), \(K\), and \(V\) tiles |
| Reuse loaded values | Reuse loaded values |
| Accumulate partial dot products | Accumulate partial attention output |
| Avoid a large broadcast intermediate | Avoid the full attention matrix |

FlashAttention adds an important complication: **online softmax**. It updates stable softmax statistics while processing score tiles. We leave the derivation and full kernel as optional advanced material.

---
## 20. Small GPU case study: composed attention versus fused SDPA

PyTorch provides `scaled_dot_product_attention`, which can dispatch to an optimized backend when the runtime supports one.

The exact backend depends on the GPU and software environment, so the benchmark result will vary.

We compare it with a straightforward composition that explicitly creates the score and probability matrices.

In [ ]:
def attention_matrix_memory_mib(
    batch: int,
    heads: int,
    sequence_length: int,
    bytes_per_element: int = 2,
):
    elements = batch * heads * sequence_length * sequence_length
    return elements * bytes_per_element / 2**20


memory_rows = []

for sequence_length in [512, 1024, 2048, 4096, 8192]:
    one_matrix_mib = attention_matrix_memory_mib(
        batch=1,
        heads=8,
        sequence_length=sequence_length,
        bytes_per_element=2,
    )

    memory_rows.append({
        "Sequence length": sequence_length,
        "One FP16 score matrix (MiB)": one_matrix_mib,
        "Scores + probabilities (MiB)": 2 * one_matrix_mib,
    })

attention_memory = pd.DataFrame(memory_rows)

attention_memory.style.format({
    "One FP16 score matrix (MiB)": "{:,.1f}",
    "Scores + probabilities (MiB)": "{:,.1f}",
})

In [ ]:
batch = 1
heads = 8
sequence_length = 1024
head_dimension = 64

shape = (
    batch,
    heads,
    sequence_length,
    head_dimension,
)

q = torch.randn(shape, device=DEVICE, dtype=torch.float16)
k = torch.randn(shape, device=DEVICE, dtype=torch.float16)
v = torch.randn(shape, device=DEVICE, dtype=torch.float16)

composed_output = composed_attention(q, k, v)
fused_output = fused_sdpa(q, k, v)

torch.testing.assert_close(
    fused_output,
    composed_output,
    rtol=5e-2,
    atol=5e-2,
)

composed_ms = benchmark_ms(lambda: composed_attention(q, k, v))
fused_ms = benchmark_ms(lambda: fused_sdpa(q, k, v))

attention_results = pd.DataFrame([
    {
        "Implementation": "Composed PyTorch attention",
        "Correct": "reference",
        "Time (ms)": composed_ms,
    },
    {
        "Implementation": "PyTorch scaled_dot_product_attention",
        "Correct": "✓",
        "Time (ms)": fused_ms,
    },
])

attention_results.style.format({
    "Time (ms)": "{:.3f}",
})

### What this case study demonstrates

The faster implementation is not using a different attention equation.

It changes how the work is organized:

- operations are fused;
- useful tiles stay close to the compute units;
- large intermediate tensors are avoided when possible; and
- data is reused before it is written back to global memory.

This is the same performance story we began with.

---
# Final Takeaways

1. **Program IDs choose the work.**
2. **Offsets choose the memory locations.**
3. **Masks protect tensor boundaries.**
4. **Tiling allows loaded data to be reused.**
5. **Large intermediates create memory traffic.**
6. **Correctness comes before performance.**
7. **Performance decisions must be measured on the target GPU.**

The progression was:

```text
vector addition
      ↓
program IDs, offsets, masks
      ↓
tiled matrix multiplication
      ↓
data reuse
      ↓
FlashAttention
```

You do not need to memorize the complete matrix-multiplication kernel today.

You should be able to look at it and explain:

- which tile one program owns;
- which data it loads;
- what it reuses;
- where it accumulates results; and
- why it uses masks.

---
# Final Takeaways

1. **Program IDs choose the work.**
2. **Offsets choose the memory locations.**
3. **Masks protect tensor boundaries.**
4. **Tiling allows loaded data to be reused.**
5. **Large intermediates create memory traffic.**
6. **Correctness comes before performance.**
7. **Performance decisions must be measured on the target GPU.**

The progression was:

```text
vector addition
      ↓
program IDs, offsets, masks
      ↓
tiled matrix multiplication
      ↓
data reuse
      ↓
FlashAttention
```

You do not need to memorize the complete matrix-multiplication kernel today.

You should be able to look at it and explain:

- which tile one program owns;
- which data it loads;
- what it reuses;
- where it accumulates results; and
- why it uses masks.

Further Reading Resources

TRITON



## C/CUDA

- [How to Optimize a CUDA Matmul Kernel for cuBLAS-like Performance: a Worklog](https://)
  
- Programming Massively Parallel Processors PAPER [VIDEO](https://www.youtube.com/watch?v=4pkbXmE4POc&list=PLRRuQYjFhpmubuwx-w8X964ofVkW1T8O4)
-